# Faruq-v3 DC2 MSFA mechanism screen

This notebook tests a final-stage transfer of DC2 Eq. (8): frozen predicted raw-RGB local features are enhanced by a GAP-pooled P5 global detector descriptor. Only the zero-initialized global projection is trainable. The locked holdout is unavailable and unopened. This is a development-validation mechanism screen, not full DC2 and not detector mAP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/dc2-msfa-screening'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('BRANCH:', BRANCH)

In [ ]:
import json, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
    'experiments/faruq-v3-dc2-predicted-raw-crop-screening-v2/dc2_predicted_raw_crop_screening.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DETECTOR = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt')
DC2B_SUMMARY = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-dc2-predicted-raw-crop-screening-v2/dc2_predicted_raw_crop_screening.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-dc2-msfa-screening-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / ('te' + 'st')).exists(), 'Locked holdout tidak boleh tersedia.'
dc2b = json.loads(DC2B_SUMMARY.read_text(encoding='utf-8'))
assert dc2b['protocol'] == 'faruq-v3-dc2-predicted-raw-crop-screening-v2'
assert dc2b['decision'] == 'PASS'
assert dc2b['next_action'] == 'AUTHORIZE_DC2_GLOBAL_LOCAL_MSFA_SCREENING'
assert dc2b['resolution'] == 128
assert dc2b['test_images_accessed'] is False
print('GPU     :', torch.cuda.get_device_name(0))
print('DETECTOR:', DETECTOR)
print('DC2b    :', DC2B_SUMMARY)
print('OUTPUT  :', OUTPUT_ROOT)

In [ ]:
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_dc2_predicted_crop.py', 'tests/test_dc2_msfa.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('PASS: zero-init identity, frozen local stream, MSFA gate, and notebook contract verified.')

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_dc2_msfa_screening',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--detector-checkpoint', str(DETECTOR),
    '--dc2b-summary', str(DC2B_SUMMARY),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--epochs', '20', '--batch-size', '64', '--workers', '2',
    '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout, end='', flush=True)
if process.returncode != 0:
    raise RuntimeError(f'DC2 MSFA screen gagal dengan return code {process.returncode}; traceback lengkap tercetak di atas.')

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'dc2_msfa_screening.json'
assert SUMMARY.is_file(), SUMMARY
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['protocol'] == 'faruq-v3-dc2-msfa-screening-v1'
assert result['evaluation_split'] == 'development_val_detector_matched_targets'
assert result['test_images_accessed'] is False
local = result['results']['dc2b_local_replay']
msfa = result['results']['msfa']['metrics']
frame = pd.DataFrame([
    {'arm': 'DC2b predicted local replay', **{k: local[k] for k in ('accuracy','macro_f1','bottom3_f1','worst_f1')}},
    {'arm': 'DC2c P5-GAP MSFA', **{k: msfa[k] for k in ('accuracy','macro_f1','bottom3_f1','worst_f1')}},
])
display(frame.style.format({'accuracy': '{:.2%}', 'macro_f1': '{:.2%}', 'bottom3_f1': '{:.2%}', 'worst_f1': '{:.2%}'}))
print('REPLAY DELTA:', result['replay_delta_vs_dc2b_report'])
print('MSFA DELTA  :', result['deltas_msfa_vs_local_only'])
print('CRITERIA    :', result['criteria'])
print('DECISION    :', result['decision'])
print('NEXT        :', result['next_action'])
print('SUMMARY     :', SUMMARY)
print('Scope: final-stage P5-GAP Eq.(8) transfer; detector/local branch frozen; not full DC2.')